# 04. UGV 팬틸트 제어 (Pan Tilt)

영상 대응: **23:47–24:34**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

`/mil/gimbal` 서비스에 절대 위치를 요청합니다.

| 필드 | 의미 | 범위 |
|---|---|---|
| pan | 좌우, 양수는 오른쪽 | -100~100 |
| tilt | 상하, 양수는 위쪽 | -100~100 |
| speed | 이동 속도 | 1~270 °/s |
| acc | 가속도 설정값 | 1~254 |

`pan` 1단위는 0.5°, `tilt` 1단위는 0.15°입니다. 위치값을 각도 자체와 혼동하지 마세요.

`ok`는 **명령 접수**, `locked`는 제어 잠금, 응답 없음은 실패 또는 시간 초과입니다. `ok`가 실제 위치 도착을 의미하지 않습니다.


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
PAN = 0.0
TILT = 0.0
SPEED = 80.0
ACC = 20.0
TIMEOUT_SECONDS = 5.0


## 실행


In [ ]:
from sim_mil_interfaces.srv import Gimbal
if not (-100 <= PAN <= 100 and -100 <= TILT <= 100 and 1 <= SPEED <= 270 and 1 <= ACC <= 254):
    raise ValueError('팬틸트 설정 범위를 확인하세요.')
node = new_node('demo_gimbal')
try:
    data, subscriptions = watch_status(node)
    require_control(data)
    client = node.create_client(Gimbal, '/mil/gimbal')
    if not client.wait_for_service(timeout_sec=TIMEOUT_SECONDS):
        raise TimeoutError('/mil/gimbal 서비스가 없습니다.')
    request = Gimbal.Request()
    request.pan, request.tilt = float(PAN), float(TILT)
    request.speed, request.acc = float(SPEED), float(ACC)
    future = client.call_async(request)
    rclpy.spin_until_future_complete(node, future, timeout_sec=TIMEOUT_SECONDS)
    if not future.done():
        future.cancel()
        raise TimeoutError('팬틸트 응답 시간 초과')
    result = future.result()
    if result is None:
        raise RuntimeError('팬틸트 응답 없음')
    print('Gimbal:', result.message)
    print('요청 위치:', PAN, TILT, '| 각도:', PAN*0.5, TILT*0.15, 'deg')
    DEMO_RESULT = {'demo':'04', 'message':result.message, 'pan':PAN, 'tilt':TILT, 'arrival_verified':False}
finally:
    node.destroy_node()


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
